# 🏋️ AI Search + Agent Service: Fitness-Fun Example 🤸

Welcome to our **AI Search + AI Agent** tutorial, where we'll:

1. **Create** an Azure AI Search index with some fitness-oriented sample data
2. **Demonstrate** how to connect that index to an Agent via the `AzureAISearchTool`
3. **Show** how to query the Agent for health and fitness info in a fun scenario (with disclaimers!)

## 🏥 Health & Fitness Disclaimer
> **This notebook is for general demonstration and entertainment purposes, NOT a substitute for professional medical advice.**
> Always seek the advice of certified health professionals.

## Prerequisites
1. Complete Agent basics notebook - [1-basics.ipynb](1-basics.ipynb)
2. An **Azure AI Search** resource (formerly "Cognitive Search"), provisioned in your Azure AI Foundry project.

## High-Level Flow
We'll do the following:
1. **Create** an AI Search index programmatically with sample fitness data.
2. **Upload** documents (fitness items) to the index.
3. **Create** an Agent that references our new index using `AzureAISearchTool`.
4. **Run queries** to see how it fetches from the index.
 
 <img src="./seq-diagrams/5-ai-search.png" width="30%"/>


## 1. Create & Populate Azure AI Search Index
We'll create a minimal index called `myfitnessindex`, containing a few example items.
Set the environment variables `SEARCH_ENDPOINT` and `SEARCH_API_KEY` (the endpoint and an admin key from your **Azure AI Search** resource in the Azure portal). We'll use the `azure.search.documents.indexes` classes to manage the index schema and upload some sample data.


In [ ]:
# Import required Azure Search libraries
import os
from azure.core.credentials import AzureKeyCredential  # For key-based authentication
from azure.search.documents.indexes import SearchIndexClient  # For managing search indexes
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, SearchableField  # Index schema components
from azure.search.documents import SearchClient  # For document operations (upload/search)

# Azure AI Search endpoint + admin key, taken from the search resource in the Azure portal.
# (Index management uses the azure-search-documents SDK, which authenticates directly to Search.)
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_credential = AzureKeyCredential(os.environ["SEARCH_API_KEY"])

# Name of our search index - this is where our fitness data will be stored
index_name = "myfitnessindex"

try:
    # SearchIndexClient manages the index itself (create/update/delete)
    index_client = SearchIndexClient(endpoint=search_endpoint, credential=search_credential)
    print("✅ Created SearchIndexClient")

    # SearchClient handles document operations (upload/search/delete documents)
    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=index_name,
        credential=search_credential,
    )
    print("✅ Created SearchClient for document operations")

except Exception as e:
    print(f"❌ Error creating search clients: {e}")

**Define the index** schema with a `FitnessItemID` key and a few fields to store product info.


In [ ]:
def create_fitness_index():
    # Define the fields (columns) for our search index
    # Each field has specific attributes that control how it can be used in searches:
    fields = [
        # Primary key field - must be unique for each document
        SimpleField(name="FitnessItemID", type=SearchFieldDataType.String, key=True),
        
        # Name field - SearchableField means we can do full-text search on it
        # filterable=True lets us filter results by name
        SearchableField(name="Name", type=SearchFieldDataType.String, filterable=True),
        
        # Category field - SearchableField for text search
        # filterable=True lets us filter by category
        # facetable=True enables category grouping in results
        SearchableField(name="Category", type=SearchFieldDataType.String, filterable=True, facetable=True),
        
        # Price field - SimpleField for numeric values
        # filterable=True enables price range filters
        # sortable=True lets us sort by price
        # facetable=True enables price range grouping
        SimpleField(name="Price", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
        
        # Description field - SearchableField for full-text search on product descriptions
        SearchableField(name="Description", type=SearchFieldDataType.String)
    ]

    # Create an index definition with our fields
    index = SearchIndex(name=index_name, fields=fields)

    # Check if index already exists - if so, delete it to start fresh
    # This is useful during development but be careful in production!
    if index_name in [x.name for x in index_client.list_indexes()]:
        index_client.delete_index(index_name)
        print(f"🗑️ Deleted existing index: {index_name}")

    # Create the new index with our schema
    created = index_client.create_index(index)
    print(f"🎉 Created index: {created.name}")

# Execute the function to create our search index
create_fitness_index()

**Upload some sample documents** to `myfitnessindex`. We'll add a few items for demonstration.


In [ ]:
def upload_fitness_docs():
    # Create a SearchClient to interact with our search index
    # This uses the endpoint + admin key we configured earlier
    search_client = SearchClient(
        endpoint=search_endpoint,
        index_name=index_name,
        credential=search_credential,
    )

    # Define sample documents that match our index schema
    # Each document must have:
    # - FitnessItemID (unique identifier)
    # - Name (searchable product name) 
    # - Category (searchable and facetable for filtering/grouping)
    # - Price (numeric field for sorting and filtering)
    # - Description (searchable product details)
    sample_docs = [
        {
            "FitnessItemID": "1",
            "Name": "Adjustable Dumbbell",
            "Category": "Strength", 
            "Price": 59.99,
            "Description": "A compact, adjustable weight for targeted muscle workouts."
        },
        {
            "FitnessItemID": "2",
            "Name": "Yoga Mat",
            "Category": "Flexibility",
            "Price": 25.0,
            "Description": "Non-slip mat designed for yoga, Pilates, and other exercises."
        },
        {
            "FitnessItemID": "3",
            "Name": "Treadmill",
            "Category": "Cardio",
            "Price": 499.0,
            "Description": "A sturdy treadmill with adjustable speed and incline settings."
        },
        {
            "FitnessItemID": "4",
            "Name": "Resistance Bands",
            "Category": "Strength",
            "Price": 15.0,
            "Description": "Set of colorful bands for light to moderate resistance workouts."
        }
    ]

    # Upload all documents to the search index in a single batch operation
    # The search service will index these documents, making them searchable
    # based on the field types we defined in our index schema
    result = search_client.upload_documents(documents=sample_docs)
    print(f"🚀 Upload result: {result}")

# Call the function to upload the documents
upload_fitness_docs()
print("✅ Documents uploaded to search index")

### Verify the documents via a basic query
Let's do a quick search for **Strength** items.


In [ ]:
# Let's verify our index by performing a basic search
# 1. First create a SearchClient using our endpoint + admin key
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=search_credential,
)

# 2. Perform a simple search query:
#    - search_text="Strength": Look for documents containing "Strength"
#    - filter=None: No additional filtering
#    - top=10: Return up to 10 matching documents
results = search_client.search(search_text="Strength", filter=None, top=10)

# 3. Print each matching document
print("🔍 Search results for 'Strength':")
print("-" * 50)
found_items = False
for doc in results:
    found_items = True
    # The doc is already a dictionary, no need for to_dict()
    print(f"Name: {doc['Name']}")
    print(f"Category: {doc['Category']}")
    print(f"Price: ${doc['Price']:.2f}")
    print(f"Description: {doc['Description']}")
    print("-" * 50)

if not found_items:
    print("No matching items found.")

## 2. Create Agent With AI Search Tool
We'll create a new agent and attach an `AzureAISearchTool` referencing **myfitnessindex**.

In your environment, you need:
- `PROJECT_ENDPOINT` — your Foundry project endpoint (`https://<resource>.services.ai.azure.com/api/projects/<project>`)
- `MODEL_DEPLOYMENT_NAME` — the deployed chat model (for example, `gpt-5.4`)
- `SEARCH_CONNECTION_NAME` — the name of the Azure AI Search **connection** in your Foundry project (created under **Operate → Admin → Connected resources**)

> **Important — grant the project access to Search.** For the agent's server-side search tool to read your index, the Foundry project's managed identity needs the **Search Index Data Contributor** and **Search Service Contributor** roles on the Azure AI Search resource (Azure portal → your Search resource → **Access control (IAM)**). Without these roles the agent is created but returns no results.

Let's initialize the `AIProjectClient` with `DefaultAzureCredential`.

In [ ]:
# Import required libraries for the agent part:
# - AIProjectClient: creates/versions agents
# - AzureAISearchTool + helpers: configure the search tool on the agent
# - PromptAgentDefinition: the v2 agent definition
import os
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AzureAISearchTool,
    PromptAgentDefinition,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
)

# Initialize the endpoint-based AIProjectClient and its OpenAI (Responses API) client.
try:
    project_client = AIProjectClient(
        endpoint=os.environ["PROJECT_ENDPOINT"],
        credential=DefaultAzureCredential(),
    )
    openai_client = project_client.get_openai_client()
    print("✅ Successfully initialized AIProjectClient + OpenAI client")
except Exception as e:
    print(f"❌ Error initializing project client: {e}")

### Find the Azure AI Search connection in your Foundry project
We'll use `project_client.connections.get(<name>)` to resolve the Azure AI Search **connection** by name and grab its **connection ID**, which the `AzureAISearchTool` needs.


In [ ]:
# Resolve the Azure AI Search connection by name and read its connection ID.
# The connection lives in your Foundry project (Management center > Connected resources).
search_connection_name = os.environ.get("SEARCH_CONNECTION_NAME")
search_connection_id = None

try:
    connection = project_client.connections.get(search_connection_name)
    search_connection_id = connection.id
    print(f"Found Azure AI Search connection '{search_connection_name}'")
    print(f"Connection ID: {search_connection_id}")
except Exception as e:
    print(f"❌ Could not resolve search connection '{search_connection_name}': {e}")

### Create the Agent with `AzureAISearchTool`
We'll attach the tool, specifying the index name we created.


In [ ]:
# Get the model deployment name from environment variables
model_name = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-5.4")
agent = None

if search_connection_id:
    # Build the Azure AI Search tool pointing at our index via the resolved connection ID.
    # query_type=SIMPLE runs keyword search (our index has no vector field).
    ai_search_tool = AzureAISearchTool(
        azure_ai_search=AzureAISearchToolResource(
            indexes=[
                AISearchIndexResource(
                    project_connection_id=search_connection_id,
                    index_name=index_name,
                    query_type=AzureAISearchQueryType.SIMPLE,
                )
            ]
        )
    )

    # Create the agent version with the AI Search tool attached.
    agent = project_client.agents.create_version(
        agent_name="fitness-agent-search",
        definition=PromptAgentDefinition(
            model=model_name,
            instructions="""
            You are a Fitness Shopping Assistant. You help users find items from the
            search index, cite what you found, but always disclaim that you don't provide
            medical advice.
            """,
            tools=[ai_search_tool],
        ),
        description="Fitness shopping assistant grounded on an Azure AI Search index.",
    )
    print(f"🎉 Created agent '{agent.name}', version: {agent.version}")

## 3. Run a Conversation with the Agent
We'll open a new conversation, post a question, and let the agent search the index for relevant items.

In [ ]:
def run_agent_query(question: str):
    # Step 1: Create a new conversation to hold the exchange
    conversation = openai_client.conversations.create()
    print(f"📝 Created conversation, ID: {conversation.id}")

    # Step 2: Run the agent on the question. A single responses.create call sends the
    # input, lets the agent use its AI Search tool, and returns the grounded answer.
    # tool_choice="required" nudges the agent to actually query the index.
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=question,
        tool_choice="required",
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
    print(f"🤖 Response status: {response.status}")

    # Step 3: Print the agent's answer
    print("\nAssistant says:")
    print(response.output_text)

    # Step 4: Surface any URL citations the agent returned from the index
    for item in (response.output or []):
        if getattr(item, "type", "") != "message":
            continue
        for content in (getattr(item, "content", None) or []):
            for ann in (getattr(content, "annotations", None) or []):
                if getattr(ann, "type", "") == "url_citation":
                    print(f"   📎 {getattr(ann, 'title', '') or ''} {getattr(ann, 'url', '')}")

# Try out our agent with two example queries:
# 1. A general question about strength training equipment
# 2. A specific request for cardio equipment with a price constraint
if agent:
    run_agent_query("Which items are good for strength training?")
    run_agent_query("I need something for cardio under $300, any suggestions?")

### 👀 See your agent live in the Foundry portal

Before deleting the agent, confirm it now exists as a first-class resource in your Foundry project:

1. In a browser, open the [Microsoft Foundry portal](https://ai.azure.com) and select your project.
2. In the top navigation select **Build**, then select **Agents** in the left pane.
3. Locate **`fitness-agent-search`** in the list — the very agent you just created from code. Open it to inspect its instructions and the attached **Azure AI Search** tool.
4. Return to this notebook and run the next cell to delete the agent and free up resources.

> **Note:** Because several notebooks each create an agent, your project can accumulate them quickly — so every notebook deletes its own agent at the end.

## 4. Cleanup
We'll clean up the agent. (In production, you might want to keep it!)

In [ ]:
if agent:
    project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
    print("🗑️ Deleted agent")

index_client.delete_index(index_name)
print(f"🗑️ Deleted index {index_name}")

# 🎉 Congrats!
You've successfully:
1. **Created** an Azure AI Search index programmatically.
2. **Populated** it with sample fitness data.
3. **Created** an Agent that queries the index using `AzureAISearchTool`.
4. **Asked** the agent for item recommendations.

Continue exploring how to integrate **OpenTelemetry** or the `azure-ai-evaluation` library for advanced tracing and evaluation capabilities. Have fun, and stay fit! 🏆